# 🎓 Projet Final : Reconnaissance d'Expressions Faciales avec Optimisation DQN

**Cours:** Deep Learning & Apprentissage par Renforcement  
**Date:** Mai 2026  
**Dataset:** FER-2013 (Facial Expression Recognition)

---

## 📋 Table des Matières

1. [Introduction et Objectifs](#1-introduction)
2. [Dataset et Prétraitement](#2-dataset)
3. [Architecture CNN pour Classification](#3-cnn)
4. [Entraînement du Modèle CNN](#4-training)
5. [Système DQN d'Optimisation](#5-dqn)
6. [Résultats et Évaluation](#6-results)
7. [Application Temps Réel](#7-realtime)
8. [Conclusion](#8-conclusion)

---

## 1. Introduction et Objectifs <a id="1-introduction"></a>

### 🎯 Objectif Principal

Ce projet implémente un système complet de reconnaissance d'expressions faciales en deux phases:

**Phase 1: Classification d'Expressions Faciales**
- Entraînement d'un CNN pour classifier 7 émotions
- Utilisation du dataset FER-2013 (35,887 images)
- Optimisation avec data augmentation

**Phase 2: Optimisation par Apprentissage par Renforcement (DQN)**
- Implémentation d'un agent DQN pour optimiser les paramètres vidéo
- Ajustement dynamique du contraste et de l'exposition
- Maximisation du score de confiance de classification

### 📊 Dataset FER-2013

- **35,887 images** en niveaux de gris (48x48 pixels)
- **7 classes d'émotions:**
  - Angry (Colère)
  - Disgust (Dégoût)
  - Fear (Peur)
  - Happy (Joie)
  - Sad (Tristesse)
  - Surprise (Surprise)
  - Neutral (Neutre)

### 🏗️ Architecture Globale du Système

```
┌─────────────────────────────────────────────────────────┐
│                    FLUX VIDÉO                           │
└────────────────────┬────────────────────────────────────┘
                     │
                     ▼
┌─────────────────────────────────────────────────────────┐
│         AJUSTEMENT DES PARAMÈTRES (DQN)                 │
│         Contraste: [0.5 - 2.0]                         │
│         Exposition: [0.5 - 2.0]                        │
└────────────────────┬────────────────────────────────────┘
                     │
                     ▼
┌─────────────────────────────────────────────────────────┐
│      CLASSIFICATION CNN (7 émotions)                    │
│      → Score de confiance                              │
└────────────────────┬────────────────────────────────────┘
                     │
                     ▼
┌─────────────────────────────────────────────────────────┐
│              AGENT DQN                                  │
│   Récompense: amélioration confiance                   │
│   Action: ajuster paramètres                           │
└─────────────────────────────────────────────────────────┘
```

## 2. Dataset et Prétraitement <a id="2-dataset"></a>

### 📦 Importation des Bibliothèques

In [ ]:
# Bibliothèques de base
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# TensorFlow et Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense, 
    Dropout, BatchNormalization, GlobalAveragePooling2D
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

# Pour le DQN
from collections import deque
import random

# Configuration
import warnings
warnings.filterwarnings('ignore')

# Style des graphiques
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Affichage de la version de TensorFlow
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU disponible: {len(tf.config.list_physical_devices('GPU')) > 0}")

### 🔧 Configuration et Constantes

In [ ]:
# Constantes du projet
IMG_DIM = 48  # Dimensions des images (48x48)
BATCH_SIZE = 64
EPOCHS = 50
LEARNING_RATE = 0.001

# Chemins des données
DATA_PATH = './'
TRAIN_PATH = os.path.join(DATA_PATH, 'train')
TEST_PATH = os.path.join(DATA_PATH, 'test')

# Classes d'émotions
CLASSES = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
NUM_CLASSES = len(CLASSES)

print(f"📊 Configuration du Projet:")
print(f"  - Dimensions des images: {IMG_DIM}x{IMG_DIM}")
print(f"  - Batch size: {BATCH_SIZE}")
print(f"  - Époques: {EPOCHS}")
print(f"  - Learning rate: {LEARNING_RATE}")
print(f"  - Nombre de classes: {NUM_CLASSES}")
print(f"  - Classes: {CLASSES}")

### 📊 Création des DataFrames

In [ ]:
def create_dataframe(data_path):
    """
    Crée un DataFrame contenant les chemins des images et leurs labels.
    
    Args:
        data_path: Chemin vers le dossier contenant les sous-dossiers de classes
    
    Returns:
        DataFrame avec colonnes 'filename' et 'class'
    """
    data = []
    for emotion_class in os.listdir(data_path):
        class_folder = os.path.join(data_path, emotion_class)
        if os.path.isdir(class_folder):
            for image_file in os.listdir(class_folder):
                if image_file.endswith('.jpg'):
                    image_path = os.path.join(class_folder, image_file)
                    data.append([image_path, emotion_class])
    
    return pd.DataFrame(data, columns=['filename', 'class'])

# Création des DataFrames
print("Chargement des données...")
df_train_full = create_dataframe(TRAIN_PATH)
df_test = create_dataframe(TEST_PATH)

# Division train/validation (80/20)
df_train, df_val = train_test_split(
    df_train_full, 
    test_size=0.2, 
    random_state=42, 
    stratify=df_train_full['class']
)

print(f"\n📊 Statistiques du dataset:")
print(f"  - Images d'entraînement: {len(df_train):,}")
print(f"  - Images de validation: {len(df_val):,}")
print(f"  - Images de test: {len(df_test):,}")
print(f"  - Total: {len(df_train) + len(df_val) + len(df_test):,}")

### 📈 Analyse Exploratoire des Données

In [ ]:
# Distribution des classes
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Distribution des Classes par Ensemble', fontsize=16, fontweight='bold', y=1.02)

# Train
df_train['class'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='skyblue', edgecolor='black')
axes[0].set_title('Entraînement', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Émotion', fontsize=12)
axes[0].set_ylabel('Nombre d\'images', fontsize=12)
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)

# Validation
df_val['class'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color='lightcoral', edgecolor='black')
axes[1].set_title('Validation', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Émotion', fontsize=12)
axes[1].set_ylabel('Nombre d\'images', fontsize=12)
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.3)

# Test
df_test['class'].value_counts().sort_index().plot(kind='bar', ax=axes[2], color='lightgreen', edgecolor='black')
axes[2].set_title('Test', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Émotion', fontsize=12)
axes[2].set_ylabel('Nombre d\'images', fontsize=12)
axes[2].tick_params(axis='x', rotation=45)
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Affichage des statistiques détaillées
print("\n📊 Distribution détaillée par classe:")
print("\n" + "="*60)
print("ENTRAÎNEMENT:")
print("="*60)
print(df_train['class'].value_counts().sort_index())
print("\n" + "="*60)
print("VALIDATION:")
print("="*60)
print(df_val['class'].value_counts().sort_index())
print("\n" + "="*60)
print("TEST:")
print("="*60)
print(df_test['class'].value_counts().sort_index())

### 🖼️ Visualisation d'Exemples d'Images

In [ ]:
# Affichage d'exemples pour chaque classe
fig, axes = plt.subplots(2, 7, figsize=(20, 6))
fig.suptitle('Exemples d\'Images par Classe d\'Émotion', fontsize=16, fontweight='bold', y=0.98)

for idx, emotion in enumerate(CLASSES):
    # Sélectionner 2 images aléatoires de chaque classe
    emotion_images = df_train[df_train['class'] == emotion].sample(min(2, len(df_train[df_train['class'] == emotion])))['filename'].values
    
    for row in range(min(2, len(emotion_images))):
        img = load_img(emotion_images[row], color_mode='grayscale', target_size=(IMG_DIM, IMG_DIM))
        axes[row, idx].imshow(img, cmap='gray')
        axes[row, idx].axis('off')
        if row == 0:
            axes[row, idx].set_title(emotion.capitalize(), fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

### 🔄 Data Augmentation et Générateurs

In [ ]:
# Générateur de données avec augmentation pour l'entraînement
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2,
    shear_range=0.2,
    fill_mode='nearest'
)

# Générateur pour validation et test (seulement normalisation)
val_test_datagen = ImageDataGenerator(rescale=1./255)

# Création des générateurs
train_generator = train_datagen.flow_from_dataframe(
    df_train,
    x_col='filename',
    y_col='class',
    target_size=(IMG_DIM, IMG_DIM),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    color_mode='grayscale',
    shuffle=True
)

val_generator = val_test_datagen.flow_from_dataframe(
    df_val,
    x_col='filename',
    y_col='class',
    target_size=(IMG_DIM, IMG_DIM),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    color_mode='grayscale',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_dataframe(
    df_test,
    x_col='filename',
    y_col='class',
    target_size=(IMG_DIM, IMG_DIM),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    color_mode='grayscale',
    shuffle=False
)

print("✅ Générateurs de données créés avec succès!")
print(f"\n📊 Informations sur les générateurs:")
print(f"  - Entraînement: {train_generator.samples} images, {len(train_generator)} batches")
print(f"  - Validation: {val_generator.samples} images, {len(val_generator)} batches")
print(f"  - Test: {test_generator.samples} images, {len(test_generator)} batches")